# Práctica: Conociendo tus datos (EDA + Tipos de atributos + Similaridad)
**Herramienta:** Python (Jupyter Notebook)  
**Tema:** Getting to Know Your Data (Data objects, estadística básica, visualización, similitud/disimilitud)

## Objetivo general
Que el estudiante cargue un dataset real, identifique **tipos de atributos**, obtenga **descripciones estadísticas**,
realice **visualizaciones** y aplique **medidas de similitud y disimilitud** (distancias) para comparar instancias.

## Entregable
En el reporte incluir:
1. Capturas/tabla con tipos de atributos y justificación breve.
2. Estadísticas (tendencia central y dispersión) para variables indicadas.
3. Visualizaciones solicitadas con interpretación corta.
4. Matrices de distancia/similitud y conclusión: ¿cuáles 2 filas son las más parecidas y por qué?

## Datasets
Análisis de un Dataset de https://archive.ics.uci.edu/datasets

> Nota: Las celdas marcadas con **ACTIVIDAD** deben completarse por el estudiante.


In [ ]:
# =========================
# 1) Librerías y configuración
# =========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# (Opcional) si tu curso ya usa scikit-learn, descomenta:
# from sklearn.preprocessing import StandardScaler
# from sklearn.metrics.pairwise import cosine_similarity


## 2) Carga del dataset

En esta práctica se recomienda el dataset **Adult Census Income (UCI)**, porque contiene variables **numéricas y categóricas**.

### Opción A (recomendada): archivo local
1) Descargue `adult.data` desde UCI.  
2) Guárdelo junto a este notebook como `adult.csv` (o use el nombre original y ajuste la ruta).  

### Opción B: URL (si hay internet habilitado)
Se puede leer desde la URL oficial.

> Si el docente proporciona otro dataset, ajuste `cols` y el nombre de archivo.


In [ ]:
# =========================
# 2.1) Leer dataset Adult
# =========================
cols = [
    "Age", "Workclass", "fnlwgt", "Education", "Education-Num",
    "Marital-status", "Occupation", "Relationship", "Race", "Sex",
    "Capital-gain", "Capital-loss", "Hours-per-week", "Native-country", "Income"
]

# Opción A: local
df = pd.read_csv("adult.csv", header=None, names=cols, skipinitialspace=True)

# Opción B: URL (descomente si aplica)
# url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
# df = pd.read_csv(url, header=None, names=cols, skipinitialspace=True)

df.head()


## 3) Limpieza mínima y validación

En este dataset, el valor `?` representa dato desconocido. Se transformará a `NaN` para poder contarlo y decidir qué hacer.

In [ ]:
# Reemplazar '?' por NaN y revisar nulos
df = df.replace("?", np.nan)

print("Tamaño original:", df.shape)
display(df.isna().sum().sort_values(ascending=False).head(10))

# Para simplificar la práctica, se eliminan filas incompletas
df = df.dropna().reset_index(drop=True)
print("Tamaño tras dropna:", df.shape)

df.head()


## 4) Objetos de datos y tipos de atributos

### 4.1 Tipos detectados por pandas


In [ ]:
df.dtypes

### 4.2 Clasificación teórica (Nominal, Binario, Ordinal, Numérico)

**ACTIVIDAD:** Complete/ajuste el diccionario `tipos` según el criterio del curso.

- **Nominal:** categorías sin orden.
- **Binario:** 2 categorías.
- **Ordinal:** categorías con orden (si se define).
- **Numérico:** valores cuantitativos.
- **Discreto vs continuo:** discreto = enteros/contable; continuo = medición con decimales (cuando aplique).


In [ ]:
# =========================
# ACTIVIDAD: ajustar clasificación teórica
# =========================
tipos = {
    "Age": "Numérico (discreto)",
    "fnlwgt": "Numérico (discreto)",
    "Education-Num": "Numérico (discreto/ordinal aproximado)",
    "Capital-gain": "Numérico (discreto)",
    "Capital-loss": "Numérico (discreto)",
    "Hours-per-week": "Numérico (discreto)",

    "Workclass": "Nominal",
    "Education": "Nominal (o Ordinal si se define un orden)",
    "Marital-status": "Nominal",
    "Occupation": "Nominal",
    "Relationship": "Nominal",
    "Race": "Nominal",
    "Native-country": "Nominal",

    "Sex": "Binario",
    "Income": "Binario"
}

tabla_tipos = pd.DataFrame({
    "Atributo": list(tipos.keys()),
    "Tipo_teorico": list(tipos.values()),
    "dtype_pandas": [df[c].dtype for c in tipos.keys()],
    "Valores_unicos": [df[c].nunique() for c in tipos.keys()]
}).sort_values("Atributo")

tabla_tipos


## 5) Descripciones estadísticas básicas

### 5.1 Estadística descriptiva general de numéricas


In [ ]:
df.describe()

### 5.2 Tendencia central (media, mediana, moda)

**ACTIVIDAD:** Calcule para `Age` y `Hours-per-week`.


In [ ]:
# =========================
# ACTIVIDAD: tendencia central
# =========================
for col in ["Age", "Hours-per-week"]:
    media = df[col].mean()
    mediana = df[col].median()
    moda = df[col].mode().iloc[0]
    print(f"{col:>15} | media={media:.3f} | mediana={mediana:.3f} | moda={moda}")


### 5.3 Dispersión (rango, cuartiles, IQR, varianza, desviación estándar)

**ACTIVIDAD:** Calcule para `Hours-per-week`.


In [ ]:
# =========================
# ACTIVIDAD: dispersión
# =========================
col = "Hours-per-week"

rango = df[col].max() - df[col].min()
q1 = df[col].quantile(0.25)
q3 = df[col].quantile(0.75)
iqr = q3 - q1
var = df[col].var()
std = df[col].std()

print("Variable:", col)
print("Rango:", rango)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Varianza:", var)
print("Desv. estándar:", std)


### 5.4 Para categóricas: frecuencias (conteos)

**ACTIVIDAD:** Muestre el top-10 de `Occupation` y `Native-country`.


In [ ]:
# =========================
# ACTIVIDAD: frecuencias
# =========================
for col in ["Occupation", "Native-country"]:
    print("\nTop 10 de:", col)
    display(df[col].value_counts().head(10))


## 6) Visualización de datos

Se realizarán visualizaciones básicas para entender:
- forma de distribución (histograma, KDE),
- dispersión y outliers (boxplot),
- relación entre variables (scatter),
- distribución de categorías (barras).


In [ ]:
# 6.1 Histograma
df["Age"].hist(bins=20)
plt.title("Histograma de Age")
plt.xlabel("Age")
plt.ylabel("Frecuencia")
plt.show()

# 6.2 Densidad (KDE)
df["Age"].plot.kde()
plt.title("Densidad (KDE) de Age")
plt.xlabel("Age")
plt.show()

# 6.3 Boxplot
df.boxplot(column=["Age", "Education-Num", "Hours-per-week"])
plt.title("Boxplot de variables numéricas")
plt.show()

# 6.4 Scatter
df.plot.scatter(x="Age", y="Hours-per-week")
plt.title("Age vs Hours-per-week")
plt.show()

# 6.5 Barras para categóricas (Top 10)
top = df["Occupation"].value_counts().head(10)
top.plot(kind="bar")
plt.title("Top 10 Occupation")
plt.xlabel("Occupation")
plt.ylabel("Frecuencia")
plt.xticks(rotation=45, ha="right")
plt.show()


## 7) Medición de similitud y disimilitud

En esta sección se comparan **filas (instancias)** del dataset.

Se trabajará con una muestra pequeña para que sea visible:


In [ ]:
muestra = df.sample(10, random_state=7).reset_index(drop=True)
muestra[["Age","Education-Num","Hours-per-week","Sex","Income","Occupation"]]


### 7.1 Distancias numéricas (Minkowski)

- p=1: Manhattan  
- p=2: Euclidiana  


In [ ]:
def minkowski(a, b, p=2):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return np.sum(np.abs(a-b)**p)**(1/p)

num_cols = ["Age", "Education-Num", "Hours-per-week"]

a = muestra.loc[0, num_cols]
b = muestra.loc[1, num_cols]

print("Fila 0:", list(a))
print("Fila 1:", list(b))
print("Manhattan (p=1):", minkowski(a, b, p=1))
print("Euclidiana (p=2):", minkowski(a, b, p=2))


### 7.2 Matriz de distancias numéricas (p=2)

**ACTIVIDAD:** Genere la matriz y visualícela como imagen.


In [ ]:
def matriz_distancias_numericas(df_small, cols, p=2):
    X = df_small[cols].to_numpy(dtype=float)
    n = X.shape[0]
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = minkowski(X[i], X[j], p=p)
    return pd.DataFrame(D)

D_euclid = matriz_distancias_numericas(muestra, num_cols, p=2)
D_euclid


In [ ]:
plt.imshow(D_euclid.values)
plt.title("Matriz de distancias (Euclidiana)")
plt.colorbar()
plt.show()


### 7.3 Distancias para binarios/nominales (Hamming y Jaccard)

Para simplificar:
- `Sex` se codifica como 1 si Male, 0 si Female.
- `Income` se codifica como 1 si >50K, 0 si <=50K.


In [ ]:
bin_df = muestra[["Sex","Income"]].copy()
bin_df["Sex"] = (bin_df["Sex"] == "Male").astype(int)
bin_df["Income"] = (bin_df["Income"] == ">50K").astype(int)
bin_df


In [ ]:
def hamming(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.mean(a != b)

def jaccard(a, b):
    a = np.array(a)
    b = np.array(b)
    inter = np.sum((a == 1) & (b == 1))
    union = np.sum((a == 1) | (b == 1))
    return 0 if union == 0 else inter / union

print("Hamming (0 vs 1):", hamming(bin_df.loc[0], bin_df.loc[1]))
print("Jaccard (0 vs 1):", jaccard(bin_df.loc[0], bin_df.loc[1]))


### 7.4 Similitud coseno (numéricas)

Se recomienda estandarizar (z-score).

In [ ]:
# Si no tienes scikit-learn, usa esta versión manual.
X = muestra[num_cols].to_numpy(dtype=float)

# Estandarización manual (z-score)
Xz = (X - X.mean(axis=0)) / X.std(axis=0)

def cosine_sim(u, v):
    u = np.array(u, dtype=float)
    v = np.array(v, dtype=float)
    denom = (np.linalg.norm(u) * np.linalg.norm(v))
    return 0 if denom == 0 else float(np.dot(u, v) / denom)

n = Xz.shape[0]
S = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        S[i, j] = cosine_sim(Xz[i], Xz[j])

S_df = pd.DataFrame(S)
S_df


In [ ]:
plt.imshow(S_df.values)
plt.title("Matriz de similitud (Coseno)")
plt.colorbar()
plt.show()


### 7.5 Distancia para tipos mixtos (estilo Gower simplificada)

Idea:
- Numéricas: distancia normalizada por rango.
- Categóricas: 0 si iguales, 1 si diferentes.
- Distancia final: promedio.

> Esta versión es una aproximación para entender el concepto de *mixed types*.


In [ ]:
num_cols = ["Age","Education-Num","Hours-per-week"]
cat_cols = ["Workclass","Marital-status","Occupation","Race","Sex"]

m = df.sample(10, random_state=3).reset_index(drop=True)

# rangos para normalizar numéricas
ranges = {c: (df[c].max() - df[c].min()) for c in num_cols}

def gower_like(row_a, row_b):
    # numéricas
    dn = 0
    for c in num_cols:
        r = ranges[c] if ranges[c] != 0 else 1
        dn += abs(row_a[c] - row_b[c]) / r
    dn /= len(num_cols)

    # categóricas
    dc = 0
    for c in cat_cols:
        dc += 0 if row_a[c] == row_b[c] else 1
    dc /= len(cat_cols)

    return (dn + dc) / 2

n = len(m)
Dmix = np.zeros((n,n))
for i in range(n):
    for j in range(n):
        Dmix[i,j] = gower_like(m.loc[i], m.loc[j])

Dmix_df = pd.DataFrame(Dmix)
Dmix_df


In [ ]:
plt.imshow(Dmix_df.values)
plt.title("Matriz de distancia mixta (Gower-like)")
plt.colorbar()
plt.show()


## 8) Actividad de comprobación (entrega)

1) **Tabla de tipos de atributos** (`tabla_tipos`).  
2) Para **Age** y **Hours-per-week**: media, mediana, moda.  
3) Para **Hours-per-week**: rango, Q1, Q3, IQR, varianza, desviación estándar.  
4) Visualizaciones obligatorias: histograma, KDE, boxplot, scatter, barras (Top-10).  
5) Con 10 filas:
   - matriz euclidiana (numéricas),
   - matriz similitud coseno (numéricas),
   - matriz Gower-like (mixta).
   - Conclusión: **¿cuáles 2 filas son las más parecidas y por qué?**

### Extra (opcional)
- Probar Minkowski con p=3.
- Incluir Capital-gain/Capital-loss y comentar el efecto de outliers.
